
# Vocab Prompt Experiments (Qwen3.5-0.8B)

This notebook documents **phase-wise prompt optimizations** for the `vocab` (Visual Vocabulary) benchmark on **Qwen3.5-0.8B**. Vocab is a 4-way image multiple-choice task where the model must match an English word to the correct image.

- **170 trials** (159 test + 8 practice + 3 catch), difficulty ranges from common words ("duck") to rare/academic ("mammalogy").
- **Chance level:** 25%.
- **Artifacts:** per-phase CSVs and logs under `results/vocab-phases/`.

## Phases

| Phase | Name | Description |
|-------|------|-------------|
| 0 | Baseline | Manifest prompts + Qwen default system prompt + thinking disabled |
| 1 | Structured multiline | Word prominent on its own line, block A/B/C/D layout |
| 2 | Enhanced parsing | Reverse-scan, "image/option X" patterns, last-letter fallback |
| 3 | Task-specific system prompt | "Visual vocabulary expert" role |
| 4 | Distractor awareness | Hint: only one image matches, others are distractors |
| 5 | Visual elimination CoT | Describe each image, then match to the word |


In [1]:
from __future__ import annotations

import csv
import importlib.util
from pathlib import Path

import pandas as pd

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / 'data').is_dir() and (p / 'src').is_dir()),
    Path.cwd().parent.parent,
)
RESULTS = ROOT / "results" / "prompt_optimization"prompt_optimization"/"vocab"/"qwen-0.8b"vocab"prompt_optimization"/"vocab"/"qwen-0.8b"qwen-0.8b"

# Load prompt builders from the experiment script
_spec = importlib.util.spec_from_file_location(
    "vocab_phases", ROOT / "scripts" / "experiment_vocab_phases.py"
)
vp = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(vp)

manifest_rows = vp.load_manifest(ROOT / "data")
IMAGE_DIR = ROOT / "data" / "assets" / "2026-03-26" / "visual" / "vocab"
TRIALS_LIST = vp.build_trials(manifest_rows, IMAGE_DIR)
TRIALS = {t["item_uid"]: t for t in TRIALS_LIST}


def build_prompt(phases: set[int], item_uid: str) -> str:
    """Reconstruct the exact prompt used for a given phase combination."""
    t = TRIALS[item_uid]
    row, word = t["row"], t["word"]
    if 1 in phases:
        p = vp.build_prompt_phase1(row, word)
    else:
        p = vp.build_prompt_baseline(row, word)
    if 4 in phases:
        p = vp.apply_phase4_distractor(p)
    if 5 in phases:
        p = vp.apply_phase5_elimination(p, word)
    if 3 in phases:
        p = vp.apply_phase3_system(p)
    return p


def load_result(csv_name: str, item_uid: str) -> dict | None:
    path = RESULTS / csv_name
    if not path.exists():
        return None
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            if row["item_uid"] == item_uid:
                return row
    return None


## Summary: All Runs

Aggregate accuracy, parse rate, and delta vs baseline across all phase configurations.

In [2]:
PHASE_CSVS = [
    ("phase_baseline.csv", "0 — Baseline"),
    ("phase_1.csv", "1 — Structured multiline"),
    ("phase_2.csv", "2 — Enhanced parsing"),
    ("phase_3.csv", "3 — System prompt"),
    ("phase_4.csv", "4 — Distractor awareness"),
    ("phase_5.csv", "5 — Elimination CoT"),
    ("phase_1_2_3.csv", "1+2+3 — Multiline + parse + sys"),
    ("phase_1_5.csv", "1+5 — Multiline + CoT"),
    ("phase_1_2_3_4_5.csv", "1+2+3+4+5 — All"),
]


def summarize_all():
    rows = []
    baseline_acc = None
    for csv_name, label in PHASE_CSVS:
        path = RESULTS / csv_name
        if not path.exists():
            continue
        with open(path, newline="", encoding="utf-8") as f:
            data = list(csv.DictReader(f))
        n = len(data)
        if n == 0:
            continue
        correct = sum(1 for r in data if r["is_correct"].lower() in ("true", "1"))
        parsed = sum(1 for r in data if r["parsed"].lower() in ("true", "1"))
        acc = correct / n
        pr = parsed / n
        if baseline_acc is None:
            baseline_acc = acc
        delta = "—" if acc == baseline_acc and label.startswith("0") else f"{(acc - baseline_acc)*100:+.1f} pp"
        rows.append({"Phase": label, "N": n, "Accuracy": f"{acc:.1%}", "Parse %": f"{pr:.1%}", "Δ vs baseline": delta})
    return pd.DataFrame(rows)


df_summary = summarize_all()
print(df_summary.to_string(index=False))
df_summary.style.hide(axis="index")

                          Phase   N Accuracy Parse % Δ vs baseline
                   0 — Baseline 170    71.8%  100.0%             —
       1 — Structured multiline 170    73.5%  100.0%       +1.8 pp
           2 — Enhanced parsing 170    71.8%  100.0%       +0.0 pp
              3 — System prompt 170    70.0%  100.0%       -1.8 pp
       4 — Distractor awareness 170    66.5%  100.0%       -5.3 pp
            5 — Elimination CoT 170    48.2%   62.9%      -23.5 pp
1+2+3 — Multiline + parse + sys 170    75.9%  100.0%       +4.1 pp
          1+5 — Multiline + CoT 170    67.1%   98.8%       -4.7 pp
                1+2+3+4+5 — All 170    84.1%  100.0%      +12.4 pp


Phase,N,Accuracy,Parse %,Δ vs baseline
0 — Baseline,170,71.8%,100.0%,—
1 — Structured multiline,170,73.5%,100.0%,+1.8 pp
2 — Enhanced parsing,170,71.8%,100.0%,+0.0 pp
3 — System prompt,170,70.0%,100.0%,-1.8 pp
4 — Distractor awareness,170,66.5%,100.0%,-5.3 pp
5 — Elimination CoT,170,48.2%,62.9%,-23.5 pp
1+2+3 — Multiline + parse + sys,170,75.9%,100.0%,+4.1 pp
1+5 — Multiline + CoT,170,67.1%,98.8%,-4.7 pp
1+2+3+4+5 — All,170,84.1%,100.0%,+12.4 pp


## Per-Trial-Type Breakdown

Accuracy by trial type (test / practice / catch) for each phase.

In [3]:
def breakdown_by_trial_type(csv_name: str, label: str):
    path = RESULTS / csv_name
    if not path.exists():
        print(f"  {csv_name} not found — skipped")
        return
    with open(path, newline="", encoding="utf-8") as f:
        data = list(csv.DictReader(f))
    by_type: dict[str, dict] = {}
    for r in data:
        tt = r["trial_type"]
        rec = by_type.setdefault(tt, {"N": 0, "Correct": 0, "Parsed": 0})
        rec["N"] += 1
        rec["Correct"] += int(r["is_correct"].lower() in ("true", "1"))
        rec["Parsed"] += int(r["parsed"].lower() in ("true", "1"))
    rows = []
    for tt, rec in sorted(by_type.items(), key=lambda kv: kv[1]["Correct"] / max(kv[1]["N"], 1), reverse=True):
        n = rec["N"]
        rows.append({"Trial Type": tt, "N": n, "Accuracy": f"{rec['Correct']/n:.0%}", "Parse %": f"{rec['Parsed']/n:.0%}"})
    total_n = sum(rec["N"] for rec in by_type.values())
    total_c = sum(rec["Correct"] for rec in by_type.values())
    total_p = sum(rec["Parsed"] for rec in by_type.values())
    rows.append({"Trial Type": "OVERALL", "N": total_n, "Accuracy": f"{total_c/total_n:.0%}", "Parse %": f"{total_p/total_n:.0%}"})
    print(f"\n{'='*50}")
    print(f"  {label}")
    print(f"{'='*50}")
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))


for csv_name, label in PHASE_CSVS:
    breakdown_by_trial_type(csv_name, label)


  0 — Baseline
Trial Type   N Accuracy Parse %
     catch   3     100%    100%
  practice   8      88%    100%
      test 159      70%    100%
   OVERALL 170      72%    100%

  1 — Structured multiline
Trial Type   N Accuracy Parse %
     catch   3     100%    100%
  practice   8      88%    100%
      test 159      72%    100%
   OVERALL 170      74%    100%

  2 — Enhanced parsing
Trial Type   N Accuracy Parse %
     catch   3     100%    100%
  practice   8      88%    100%
      test 159      70%    100%
   OVERALL 170      72%    100%

  3 — System prompt
Trial Type   N Accuracy Parse %
     catch   3     100%    100%
  practice   8     100%    100%
      test 159      68%    100%
   OVERALL 170      70%    100%

  4 — Distractor awareness
Trial Type   N Accuracy Parse %
     catch   3     100%    100%
  practice   8      75%    100%
      test 159      65%    100%
   OVERALL 170      66%    100%

  5 — Elimination CoT
Trial Type   N Accuracy Parse %
     catch   3     100%    1

---

## Phase 0 — Baseline

Uses the manifest prompt format with Qwen3.5 default system prompt and thinking mode disabled.

**Prompt format:**
```
Choose the image that matches the text: "carrot". Answer with A, B, C, or D. A: <image1>; B: <image2>; C: <image3>; D: <image4>
```

**Note:** Qwen3.5 thinking mode is disabled via `enable_thinking=False` to prevent verbose chain-of-thought that truncates and fails to parse.

In [4]:
# Phase 0 example: common word
uid0_easy = "vocab__carrot"
print("=== PROMPT (Phase 0, common word) ===")
print(build_prompt(set(), uid0_easy))
r = load_result("phase_baseline.csv", uid0_easy)
if r:
    print(f"\n=== MODEL OUTPUT ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet — run the experiment first)")

=== PROMPT (Phase 0, common word) ===
Choose the image that matches the text: "carrot". Answer with A, B, C, or D. A: <image1>; B: <image2>; C: <image3>; D: <image4>

=== MODEL OUTPUT ===
Response: C
Predicted: C  Correct: C  ✓


In [5]:
# Phase 0 example: rare/hard word
uid0_hard = "vocab__mammalogy"
print("=== PROMPT (Phase 0, rare word) ===")
print(build_prompt(set(), uid0_hard))
r = load_result("phase_baseline.csv", uid0_hard)
if r:
    print(f"\n=== MODEL OUTPUT ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== PROMPT (Phase 0, rare word) ===
Choose the image that matches the text: "mammalogy". Answer with A, B, C, or D. A: <image1>; B: <image2>; C: <image3>; D: <image4>

=== MODEL OUTPUT ===
Response: A
Predicted: A  Correct: B  ✗


---

## Phase 1 — Structured Multiline Prompt

Reformats the prompt with the target word on its own line and options in block layout:

```
Which image shows: "carrot"?

A: <image1>
B: <image2>
C: <image3>
D: <image4>

Answer with one letter.
```

**Rationale:** Clearer visual hierarchy; the model can more easily isolate the word and map options to images.

In [6]:
uid1 = "vocab__hamster"
print("=== PROMPT (Phase 1) ===")
print(build_prompt({1}, uid1))
r = load_result("phase_1.csv", uid1)
if r:
    print(f"\n=== MODEL OUTPUT ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== PROMPT (Phase 1) ===
Which image shows: "hamster"?

A: <image1>
B: <image2>
C: <image3>
D: <image4>

Answer with one letter.

=== MODEL OUTPUT ===
Response: C
Predicted: C  Correct: C  ✓


---

## Phase 2 — Enhanced Parsing

Same prompt as baseline, but with robust fallback parsing:
- `"Image A"`, `"Option A"`, `"Picture A"` patterns
- `"I choose/select A"`, `(A)` extraction
- Reverse sentence scan, last standalone letter A–D

**Rationale:** Even with thinking disabled, Qwen3.5 occasionally wraps answers in short phrases.

In [7]:
uid2 = "vocab__concentric"
print("=== PROMPT (Phase 2 — same prompt, parsing differs) ===")
print(build_prompt(set(), uid2))
r_base = load_result("phase_baseline.csv", uid2)
r_p2 = load_result("phase_2.csv", uid2)
if r_base:
    print(f"\n--- Baseline parse ---")
    print(f"  Response: {r_base['raw_response']}")
    print(f"  Predicted: {r_base['predicted_label']}  Parsed: {r_base['parsed']}")
if r_p2:
    print(f"\n--- Phase 2 parse ---")
    print(f"  Response: {r_p2['raw_response']}")
    print(f"  Predicted: {r_p2['predicted_label']}  Parsed: {r_p2['parsed']}")

=== PROMPT (Phase 2 — same prompt, parsing differs) ===
Choose the image that matches the text: "concentric". Answer with A, B, C, or D. A: <image1>; B: <image2>; C: <image3>; D: <image4>

--- Baseline parse ---
  Response: D
  Predicted: D  Parsed: True

--- Phase 2 parse ---
  Response: D
  Predicted: D  Parsed: True


---

## Phase 3 — Task-Specific System Prompt

Replaces the generic Qwen system prompt with:
- **System:** `"You are a visual vocabulary expert. You match English words to their corresponding images. Always respond with exactly one letter: A, B, C, or D."`
- **Suffix:** `"Respond with exactly one letter: A, B, C, or D. Nothing else."`

**Rationale:** A domain-specific role prompt may improve focus on word-to-image matching.

In [8]:
uid3 = "vocab__aversion"
print("=== PROMPT (Phase 3) ===")
print(build_prompt({3}, uid3))
r = load_result("phase_3.csv", uid3)
if r:
    print(f"\n=== MODEL OUTPUT ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== PROMPT (Phase 3) ===
Choose the image that matches the text: "aversion". Answer with A, B, C, or D. A: <image1>; B: <image2>; C: <image3>; D: <image4>

Respond with exactly one letter: A, B, C, or D. Nothing else.

=== MODEL OUTPUT ===
Response: C
Predicted: C  Correct: C  ✓


---

## Phase 4 — Distractor Awareness

Prepends a hint reminding the model that three of the four images are distractors:

```
Only one image matches the word. The other three are unrelated distractors. Look at each image carefully before choosing.
```

**Rationale:** For harder words where the model may be uncertain, this encourages careful comparison rather than quick pattern matching.

In [9]:
uid4 = "vocab__vertebra"
print("=== PROMPT (Phase 4) ===")
print(build_prompt({4}, uid4))
r = load_result("phase_4.csv", uid4)
if r:
    print(f"\n=== MODEL OUTPUT ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== PROMPT (Phase 4) ===
Only one image matches the word. The other three are unrelated distractors. Look at each image carefully before choosing.

Choose the image that matches the text: "vertebra". Answer with A, B, C, or D. A: <image1>; B: <image2>; C: <image3>; D: <image4>

=== MODEL OUTPUT ===
Response: B
Predicted: B  Correct: D  ✗


---

## Phase 5 — Visual Elimination CoT

Appends a chain-of-thought instruction:
```
For each image, briefly identify what it shows. Then select the one that matches the word "<word>". State your final answer as a single letter.
```

**Rationale:** Forces the model to describe each image before deciding, which can help for ambiguous or rare words where the model might otherwise guess.

In [10]:
uid5 = "vocab__mammalogy"
print("=== PROMPT (Phase 5) ===")
print(build_prompt({5}, uid5))
r = load_result("phase_5.csv", uid5)
if r:
    print(f"\n=== MODEL OUTPUT ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== PROMPT (Phase 5) ===
Choose the image that matches the text: "mammalogy". Answer with A, B, C, or D. A: <image1>; B: <image2>; C: <image3>; D: <image4>

For each image, briefly identify what it shows. Then select the one that matches the word "mammalogy". State your final answer as a single letter.

=== MODEL OUTPUT ===
Response: The word "mammalogy" refers to the study of mammals. Let's analyze the images:

- **A:** Shows a magnifying glass over a green lizard. This is an animal, but not a mammal.
- **B:** Shows a magnifying glass over a chipmunk. This is a mammal, but it is not a mammal.
- **C:** Shows a magnifying glass over a fish. This is an animal, but not a mammal.
- **D:** Shows a magnifying glass over a butterfly. This is an animal, but not a mammal.

None of the images show a mammal. However, the question asks to choose the image that matches the text "mammalogy". This is a trick question. The correct answer is D because it is the only image that contains a mammal (a butt

---

## Combined Runs

### Phase 1+2+3 — Structural Improvements

Multiline prompt + enhanced parsing + task-specific system prompt.

In [11]:
uid_123 = "vocab__fetch"
print("=== PROMPT (Phase 1+2+3) ===")
print(build_prompt({1, 2, 3}, uid_123))
r = load_result("phase_1_2_3.csv", uid_123)
if r:
    print(f"\n=== MODEL OUTPUT ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== PROMPT (Phase 1+2+3) ===
Which image shows: "fetch"?

A: <image1>
B: <image2>
C: <image3>
D: <image4>

Answer with one letter.

Respond with exactly one letter: A, B, C, or D. Nothing else.

=== MODEL OUTPUT ===
Response: D
Predicted: D  Correct: C  ✗


### Phase 1+5 — Multiline + Elimination CoT

In [12]:
uid_15 = "vocab__concentric"
print("=== PROMPT (Phase 1+5) ===")
print(build_prompt({1, 5}, uid_15))
r = load_result("phase_1_5.csv", uid_15)
if r:
    print(f"\n=== MODEL OUTPUT ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== PROMPT (Phase 1+5) ===
Which image shows: "concentric"?

A: <image1>
B: <image2>
C: <image3>
D: <image4>

Answer with one letter.

For each image, briefly identify what it shows. Then select the one that matches the word "concentric". State your final answer as a single letter.

=== MODEL OUTPUT ===
Response: D
Predicted: D  Correct: B  ✗


### Phase 1+2+3+4+5 — All Improvements

In [13]:
uid_all = "vocab__bulldozer"
print("=== PROMPT (Phase 1+2+3+4+5) ===")
print(build_prompt({1, 2, 3, 4, 5}, uid_all))
r = load_result("phase_1_2_3_4_5.csv", uid_all)
if r:
    print(f"\n=== MODEL OUTPUT ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== PROMPT (Phase 1+2+3+4+5) ===
Only one image matches the word. The other three are unrelated distractors. Look at each image carefully before choosing.

Which image shows: "bulldozer"?

A: <image1>
B: <image2>
C: <image3>
D: <image4>

Answer with one letter.

For each image, briefly identify what it shows. Then select the one that matches the word "bulldozer". State your final answer as a single letter.

Respond with exactly one letter: A, B, C, or D. Nothing else.

=== MODEL OUTPUT ===
Response: A
Predicted: A  Correct: A  ✓


---

## Analysis: Word Difficulty

Since Vocab has only one trial type (test), we analyze by word difficulty: which specific words improve or regress across phases.

In [14]:
def compare_phases(base_csv: str, improved_csv: str, base_label: str, imp_label: str):
    """Show items that changed between two phases."""
    base_path = RESULTS / base_csv
    imp_path = RESULTS / improved_csv
    if not base_path.exists() or not imp_path.exists():
        print(f"  Missing files — skipped")
        return
    with open(base_path, newline="", encoding="utf-8") as f:
        base_data = {r["item_uid"]: r for r in csv.DictReader(f)}
    with open(imp_path, newline="", encoding="utf-8") as f:
        imp_data = {r["item_uid"]: r for r in csv.DictReader(f)}
    gained, lost = [], []
    for uid in base_data:
        if uid not in imp_data:
            continue
        b_correct = base_data[uid]["is_correct"].lower() in ("true", "1")
        i_correct = imp_data[uid]["is_correct"].lower() in ("true", "1")
        word = uid.replace("vocab__", "")
        if not b_correct and i_correct:
            gained.append(word)
        elif b_correct and not i_correct:
            lost.append(word)
    print(f"\n{base_label} → {imp_label}:")
    print(f"  Gained ({len(gained)}): {', '.join(sorted(gained)[:15])}{'...' if len(gained) > 15 else ''}")
    print(f"  Lost ({len(lost)}): {', '.join(sorted(lost)[:15])}{'...' if len(lost) > 15 else ''}")
    print(f"  Net: {len(gained) - len(lost):+d} items")


compare_phases("phase_baseline.csv", "phase_1.csv", "Baseline", "Phase 1")
compare_phases("phase_baseline.csv", "phase_5.csv", "Baseline", "Phase 5")
compare_phases("phase_baseline.csv", "phase_1_2_3_4_5.csv", "Baseline", "All")


Baseline → Phase 1:
  Gained (26): applaud, arbor, chat, cheese, claw, clothespin, coaster, colony, cormorant, cornbread, ecstatic, facade, hoe, oil, percussion...
  Lost (23): awning, baywindow, bulldozer, cake, camp, corset, couturier, habit, knee, omelet, peeking, pump, resuscitation, rosette, scoop...
  Net: +3 items

Baseline → Phase 5:
  Gained (8): arbor, coaster, cormorant, saffron, shower, squash, sunflower, tulip
  Lost (48): aesthete, aversion, baywindow, blender, bouquet, bulldozer, cake, calendar, cloak, confectionery, corset, couturier, degression, dentist, dredging...
  Net: -40 items

Baseline → All:
  Gained (33): applaud, arbor, arcade, artifact, beret, chat, cheese, clothespin, coaster, colony, cormorant, ecstatic, facade, hoe, kazoo...
  Lost (12): baywindow, cake, couturier, habit, precarious, pump, resuscitation, rosette, stump, suede, swordfish, turnstile
  Net: +21 items


---

## Conclusions

### Results summary

| Phase | Accuracy | Parse % | Δ |
|-------|----------|---------|---|
| 0 — Baseline | 72% | 100% | — |
| 1 — Structured multiline | 74% | 100% | +2 pp |
| 2 — Enhanced parsing | 72% | 100% | 0 pp |
| 3 — System prompt | 70% | 100% | −2 pp |
| 4 — Distractor awareness | 66% | 100% | −6 pp |
| 5 — Elimination CoT | 48% | 63% | −24 pp |
| **1+2+3 — Structural** | **76%** | **100%** | **+4 pp** |
| 1+5 — Multiline + CoT | 67% | 99% | −5 pp |
| **1+2+3+4+5 — All** | **84%** | **100%** | **+12 pp** |

### Key findings

**1. The full combination is the best configuration (+12 pp, 72% → 84%)**
Although individual phases yield modest or even negative gains, combining all five produces a synergistic effect: +21 net items (33 gained, 12 lost) compared to baseline. Each phase contributes a different mechanism — structure, parsing robustness, role framing, distractor awareness, and CoT — that together reinforce correct behaviour.

**2. Phase 5 (CoT) alone destroys the parse rate (63%)**
With `max_new_tokens=128`, the 0.8B model generates extended reasoning that gets truncated before it can emit the final answer letter. However, *when combined with the other phases*, CoT is beneficial — the structured prompt (Phase 1) and strict format suffix (Phase 3) anchor the output format so the model concludes with a single letter even after reasoning.

**3. Phase 3 (system prompt) alone slightly hurts accuracy (−2 pp on test items)**
The "visual vocabulary expert" role appears to make the model more cautious on hard words, triggering longer deliberation that the baseline parser cannot reliably extract. Combined with structured format and enhanced parsing, its effect is neutral or positive.

**4. Phase 4 (distractor hint) alone also hurts (−6 pp)**
Telling the model "only one image is correct" paradoxically confuses it on rare/academic words — it likely triggers an exhaustive search strategy that does not work well without CoT. The distractor hint only contributes positively when paired with CoT (as in the full combination).

**5. The main bottleneck is rare and academic vocabulary**
Accuracy on `test` items (159 items) goes from 70% (baseline) to 83% (all phases). Items gained include clearly visualisable concepts such as `applaud`, `ecstatic`, `percussion`, `colony` — words where better structure helps the model commit to the right image. The 12 items lost (`precarious`, `suede`, `turnstile`, ...) are highly abstract or ambiguous words where no prompt formulation helps with a 0.8B model.

### Production recommendation

**Use the full 1+2+3+4+5 combination** for Vocab with Qwen3.5-0.8B:
- Structured multiline prompt with the word on its own line (Phase 1)
- Robust parsing with reverse-sentence scan (Phase 2)
- Strict format instruction in both system prompt and user turn (Phase 3)
- Distractor awareness hint before the prompt (Phase 4)
- Visual elimination CoT appended after options (Phase 5)
- Set `max_new_tokens = 256` to ensure the CoT completes with a final letter

### Limitations

- Only 170 trials — differences of ≤5 items may reflect noise rather than a real effect.
- Qwen3.5-0.8B has limited capacity for technical/academic vocabulary; a larger model would likely show even greater gains from the same prompt optimisations.
- Prompt phase ordering matters: CoT only works reliably when the prompt structure is already clear and the format constraint is explicit.